## __Phase 3: Modeling__

### __Import packages__

In [1]:
import pandas as pd

### __Load forward results__

In [14]:
forward_results = pd.read_pickle("../scripts/test_scripts/alignments_df_new.pkl")
forward_results.head(5)

,RF_signal_id,RF_track_id,AIS_track_id,forward_score,normalized_forward_score,is_true_match
0,1,"(205717000, 3)","(205717000, 3)",-16.953044,-0.044849,True
1,2,"(205717000, 3)","(205717000, 3)",-16.620944,-0.043971,True
2,3,"(205717000, 3)","(205717000, 3)",-15.899161,-0.042061,True
3,4,"(205717000, 3)","(205717000, 3)",-15.678727,-0.041478,True
4,1,"(205717000, 7)","(205717000, 7)",-11.101144,-0.055785,True


### __Summary statistics of forward results__

In [17]:
# Count how many unique AIS track IDS are linked to each RF signal 
AIS_tracks_per_RF_signal = forward_results.groupby(["RF_signal_id", "RF_track_id"])["AIS_track_id"].nunique()

# Calculate the average number of AIS tracks linked to each RF track 
average_AIS_matches_per_RF_signal = AIS_tracks_per_RF_signal.mean()
print(f"Average number of AIS tracks per RF track: {average_AIS_matches_per_RF_signal:.2f}")

Average number of AIS tracks per RF track: 2.56


In [30]:
best_predictions = forward_results.loc[
    forward_results.groupby(["RF_signal_id", "RF_track_id"])["normalized_forward_score"].idxmax()
]

# TP: a correct match between an RF signal and AIS track
# FP: an incorrect match, the algorithm believes it is a match, but in reality is not
# Precision: TP / (TP + FP)

TP = best_predictions["is_true_match"].sum()
FP = (~best_predictions["is_true_match"]).sum()
precision = TP / (TP + FP) 
print(f"Precision: {precision:.4f}")

Precision: 0.9009


### __Experiment 1__

In [94]:
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.viterbi_algorithm.states import *

states = [ BeginState(), AISState(), RFState(), MState(), EndState()]
viterbi_model = PHMM_viterbi(states, EndState())

In [95]:
# Analysis of true vs. incorrect matches for the same RF signal 

# 1. Select all incorrectly predicted matches (i.e., mismatches)
mismatches = best_predictions[best_predictions["is_true_match"] == False]

# 2. Create a set of (RF_track_id, RF_signal_id) pairs to identify mismatches
mismatch_keys = set(zip(mismatches['RF_track_id'], mismatches['RF_signal_id']))

# 3. From all forward results, find the correct (true) matches for the same RF signals that were incorrectly matched
true_matches = forward_results[
    (forward_results['is_true_match'] == True) &
    forward_results.apply(lambda row: (row['RF_track_id'], row['RF_signal_id']) in mismatch_keys, axis=1)
]

# 4. Merge both so that we have the best prediction and the ground trust per RF signal
df_experiment_one = mismatches.merge(
    true_matches,
    on = ["RF_signal_id", "RF_track_id"],
    suffixes=("_pred", "_true")
)

df_experiment_one.head(5)

,RF_signal_id,RF_track_id,AIS_track_id_pred,forward_score_pred,normalized_forward_score_pred,is_true_match_pred,AIS_track_id_true,forward_score_true,normalized_forward_score_true,is_true_match_true
0,1,"(209087000, 10)","(351567000, 19)",-33.793927,-0.031465,False,"(209087000, 10)",-28.996889,-0.032291,True
1,1,"(210499000, 15)","(368270390, 0)",-129.616274,-0.029272,False,"(210499000, 15)",-22.094063,-0.033425,True
2,1,"(215228000, 7)","(636017316, 12)",-36.448784,-0.031126,False,"(215228000, 7)",-13.536929,-0.046519,True
3,1,"(215765000, 2)","(636019582, 6)",-49.652530,-0.029207,False,"(215765000, 2)",-38.198289,-0.030364,True
4,1,"(235010180, 19)","(477909100, 31)",-27.748794,-0.035484,False,"(235010180, 19)",-21.887682,-0.036238,True


In [96]:
df_subset = pd.read_pickle("../scripts/test_scripts/sample_set_5000.pkl")
df_subset.head(5)

,ID,TrackID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
382,"(205717000, 3)",3,205717000,2024-01-01 03:40:23,NaT,NaT,None,None,begin
4,"(205717000, 3)",3,205717000,2024-01-01 03:40:24,2024-01-01 03:40:24,NaT,"[46.89339, -125.42151]",None,AIS
5,"(205717000, 3)",3,205717000,2024-01-01 03:44:15,2024-01-01 03:44:15,NaT,"[46.89344, -125.39996]",None,AIS
6,"(205717000, 3)",3,205717000,2024-01-01 03:45:33,2024-01-01 03:45:33,NaT,"[46.8935, -125.39274]",None,AIS
7,"(205717000, 3)",3,205717000,2024-01-01 03:46:39,2024-01-01 03:46:39,NaT,"[46.89348, -125.38572]",None,AIS


In [97]:
results = []

# Group the training data so that we can quickly retrieve sequences per route
df_grouped = df_subset.groupby("ID")

df_RF = df_subset[df_subset["RF"].notna()].copy()
df_RF["RF_signal_id"] = df_RF.groupby("ID").cumcount() + 1

def get_AIS_seq(df_grouped, track_id):
    AIS_data = df_grouped.get_group(track_id)
    return [
        (dt, coord) for dt, coord in zip(AIS_data["AIS_Timestamp"], AIS_data["AIS"])
        if pd.notna(dt) and coord is not None
    ]
    
def get_closest_AIS_in_time(AIS_seq, RF_datetime):
    return min(AIS_seq, key=lambda x: abs((x[0] - RF_datetime).total_seconds()))

for _, row in df_experiment_one.iterrows():
    RF_signal_id = row["RF_signal_id"]
    RF_track_id = row["RF_track_id"]
    
    RF_signal_info = df_RF[(df_RF["RF_signal_id"] == RF_signal_id) & (df_RF["ID"] == RF_track_id)].iloc[0]
    RF_datetime = RF_signal_info["RF_Timestamp"]
    RF_coordinates = RF_signal_info["RF"]
    RF_seq = [(RF_datetime, RF_coordinates)]
    
    AIS_seq_pred = get_AIS_seq(df_grouped, row["AIS_track_id_pred"])
    AIS_seq_true = get_AIS_seq(df_grouped, row["AIS_track_id_true"])
    
    #  Find nearest AIS point in time (may be before or after RF)
    true_dt_nearest, true_coord_nearest = get_closest_AIS_in_time(AIS_seq_true, RF_datetime)
    pred_dt_nearest, pred_coord_nearest = get_closest_AIS_in_time(AIS_seq_pred, RF_datetime)

    # Calculate difference in time and distance
    distance_pred = haversine(RF_coordinates, pred_coord_nearest)
    distance_true = haversine(RF_coordinates, true_coord_nearest)

    time_diff_pred = abs((RF_datetime - pred_dt_nearest).total_seconds())
    time_diff_true = abs((RF_datetime - true_dt_nearest).total_seconds()) 
    
    # Calculate viterbi score
    path_pred, score_pred, log_probs_pred = viterbi_model.viterbi(AIS_seq_pred, RF_seq)
    path_true, score_true, log_probs_true = viterbi_model.viterbi(AIS_seq_true, RF_seq)

    # Save results
    results.append({
        "RF_signal_id": RF_signal_id,
        "RF_track_id": RF_track_id,
        "AIS_track_id_pred": row["AIS_track_id_pred"],
        "AIS_track_id_true": row["AIS_track_id_true"],
        "forward_score_pred": row["forward_score_pred"],
        "forward_score_true": row["forward_score_true"],
        "normalized_forward_score_pred": row["normalized_forward_score_pred"],
        "normalized_forward_score_true": row["normalized_forward_score_true"],
        "distance_diff_pred": distance_pred,
        "distance_diff_true": distance_true,
        "time_diff_pred": time_diff_pred,
        "time_diff_true": time_diff_true,
        "viterbi_score_pred": score_pred,
        "viterbi_score_true": score_true,
        "normalized_viterbi_score_pred": score_pred/len(AIS_seq_pred),
        "normalized_viterbi_score_true": score_true/len(AIS_seq_true),
        "path_pred": path_pred,
        "path_true": path_true,
    })
    
    break

In [98]:
pd.DataFrame(results)

,RF_signal_id,RF_track_id,AIS_track_id_pred,AIS_track_id_true,forward_score_pred,forward_score_true,normalized_forward_score_pred,normalized_forward_score_true,distance_diff_pred,distance_diff_true,time_diff_pred,time_diff_true,viterbi_score_pred,viterbi_score_true,normalized_viterbi_score_pred,normalized_viterbi_score_true,path_pred,path_true
0,1,"(209087000, 10)","(351567000, 19)","(209087000, 10)",-33.793927,-28.996889,-0.031465,-0.032291,2.982666,2.014827,31.0,1.0,-33.793927,-28.996889,-0.031465,-0.032291,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...","[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [ ]:
# Haal dichtstbijzijnde AIS-punt in tijd (kan voor of na RF liggen)
true_dt_nearest, true_coord_nearest = get_closest_AIS_in_time(AIS_seq_true, RF_datetime)
pred_dt_nearest, pred_coord_nearest = get_closest_AIS_in_time(AIS_seq_pred, RF_datetime)

# Bereken afstand en tijdsverschil
distance_true = haversine(RF_coordinates, true_coord_nearest)
distance_pred = haversine(RF_coordinates, pred_coord_nearest)

time_diff_true = abs((RF_datetime - true_dt_nearest).total_seconds()) 
time_diff_pred = abs((RF_datetime - pred_dt_nearest).total_seconds())
